In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable


In [0]:
%run /Workspace/consolidated_pipeline/Setup/Utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source", "customers","Data Source")



In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://child-company-dp/{data_source}/*.csv'
print(base_path)


s3://child-company-dp/customers/*.csv


In [0]:
df =(
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", f.current_timestamp())
    .select("*","_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))


customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-07-18T15:11:49.291Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-07-18T15:11:49.291Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-07-18T15:11:49.291Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-07-18T15:11:49.291Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-07-18T15:11:49.291Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-07-18T15:11:49.291Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-07-18T15:11:49.291Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-07-18T15:11:49.291Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-07-18T15:11:49.291Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-07-18T15:11:49.291Z,customers.csv,1404


In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-07-18 15:11:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-07-18 15:11:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-07-18 15:11:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-07-18 15:11:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-07-18 15:11:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-07-18 15:11:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-07-18 15:11:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-07-18 15:11:...|customers.csv|     1404|
|     7891

In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])


In [0]:
display(
    df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-07-18T15:11:51.825Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-07-18T15:11:51.825Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-07-18T15:11:51.825Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn("customer_name", f.trim(f.col("customer_name")))


In [0]:
df_silver.select("city").distinct().show()


+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
city_mapping = {
    "Bengaluruu": "Bengaluru",
    "Bengalore": "Bengaluru",

    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",

    "NewDelhee": "New Delhi",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi"
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = df_silver.replace(city_mapping,subset=["city"]).withColumn("city", 
    f.when(f.col("city").isNull(), None)
    .when(f.col("city").isin(allowed),f.col("city"))
    .otherwise(None)
)

In [0]:
df_silver.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      FitFuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|HydroBoost Nutrition|
|MacroBite Superfoods|
|MacroBite superfoods|
|      PowerSnack Hub|
|      PowerSnack hub|
|   SprintX nutrition|
|   SprintX Nutrition|
|    ZenAthlete foods|
|    ZenAthlete Foods|
|Peak performance ...|
|Peak Performance ...|
| PrimeFuel Nutrition|
|       Recovery Lane|
|      StaminaX Store|
|EliteAthlete Nutr...|
|      GamePlan Foods|
|   Champion's choice|
+--------------------+
only showing top 20 rows


In [0]:
df_silver = df_silver.withColumn("customer_name", 
    f.when(f.col("customer_name").isNull(), None)
    .otherwise(f.initcap("customer_name"))
)

df_silver.select("customer_name").distinct().show()


+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=False)

+-----------+-------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name      |city|read_timestamp            |file_name    |file_size|
+-----------+-------------------+----+--------------------------+-------------+---------+
|789403     |Sprintx Nutrition  |NULL|2026-07-18 15:11:51.825361|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL|2026-07-18 15:11:51.825361|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL|2026-07-18 15:11:51.825361|customers.csv|1404     |
|789603     |Recovery Lane      |NULL|2026-07-18 15:11:51.825361|customers.csv|1404     |
+-----------+-------------------+----+--------------------------+-------------+---------+



In [0]:
customer_city_fix = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame([(k,v) for k,v in customer_city_fix.items()], ["customer_id", "Fixed_city"])


In [0]:
df_silver = (
    df_silver
    .join(df_fix,"customer_id","left")
    .withColumn("city", f.coalesce("city", "Fixed_city"))
    .drop("Fixed_city")
)
display(df_silver)


customer_id,customer_name,city,read_timestamp,file_name,file_size
789622,Eliteathlete Nutrition,New Delhi,2026-07-18T15:11:51.825Z,customers.csv,1404
789321,Powersnack Hub,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404
789601,Recovery Lane,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789720,Gameplan Foods,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789201,Fitfuel Market,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789420,Zenathlete Foods,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404
789202,Fitfuel Market,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404
789122,Hydroboost Nutrition,New Delhi,2026-07-18T15:11:51.825Z,customers.csv,1404
789421,Zenathlete Foods,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn("customer_id", f.col("customer_id").cast("string"))

print(df_silver.printSchema())



root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer", f.concat_ws("-","customer_name",f.coalesce("city"), f.lit("Unknown"))
    )


    .withColumn("market", f.lit("India"))
    .withColumn("platform", f.lit("Sports Bar"))
    .withColumn("channel", f.lit("Acquisition"))
    
)
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-07-18T15:11:51.825Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi-Unknown,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-07-18T15:11:51.825Z,customers.csv,1404,Powersnack Hub-Hyderabad-Unknown,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404,Recovery Lane-Bengaluru-Unknown,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404,Gameplan Foods-Bengaluru-Unknown,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-07-18T15:11:51.825Z,customers.csv,1404,Fitfuel Market-Bengaluru-Unknown,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.dim_customers").select(
    f.col("customer_id").alias("customer_code"),"customer","market","platform","channel"
)



In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.gold.dim_customers")
df_child_customers = df_silver.select(
    f.col("customer_id").alias("customer_code"),"customer","market","platform","channel"
)

delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]